In [79]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# Load the dataset
data = pd.read_csv('master_data_imputed.csv')
data.head()
data.columns# Summary statistics    

C:\Users\DELL\AppData\Local\Temp\ipykernel_2972\2621632569.py:5: DtypeWarning: Columns (0: stress_level) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('master_data_imputed.csv')


Index(['user_id', 'age', 'gender', 'screen_time_hours', 'sleep_hours',
       'stress_level', 'tech_usage_hours', 'social_media_hours',
       'gaming_hours', 'mental_health_status', 'physical_activity_hours',
       'support_systems_access', 'work_environment_impact',
       'online_support_usage', 'daily_exercise_mins', 'diet_quality',
       'productivity_score', 'mood_level', 'occupation', 'work_mode',
       'work_screen_hours', 'leisure_screen_hours', 'sleep_quality',
       'exercise_minutes_per_week', 'social_hours_per_week',
       'mental_wellness_index', 'user_archetype', 'primary_platform',
       'dominant_content_type', 'activity_type', 'late_night_usage',
       'social_comparison_trigger', 'gad7_score', 'gad7_severity',
       'phq9_score', 'phq9_severity'],
      dtype='str')

In [41]:
data.columns

Index(['user_id', 'age', 'gender', 'screen_time_hours', 'sleep_hours',
       'stress_level', 'tech_usage_hours', 'social_media_hours',
       'gaming_hours', 'mental_health_status', 'physical_activity_hours',
       'support_systems_access', 'work_environment_impact',
       'online_support_usage', 'daily_exercise_mins', 'diet_quality',
       'productivity_score', 'mood_level', 'occupation', 'work_mode',
       'work_screen_hours', 'leisure_screen_hours', 'sleep_quality',
       'exercise_minutes_per_week', 'social_hours_per_week',
       'mental_wellness_index', 'user_archetype', 'primary_platform',
       'dominant_content_type', 'activity_type', 'late_night_usage',
       'social_comparison_trigger', 'gad7_score', 'gad7_severity',
       'phq9_score', 'phq9_severity'],
      dtype='str')

In [80]:
# ================================
# 1. Nettoyage des données texte
# ================================
data['gender'] = data['gender'].str.strip()

# ================================
# 2. Encodage des variables
# ================================

# 2.1 Gender → encoding catégoriel
data['gender'] = data['gender'].astype('category').cat.codes

# 2.2 Mental health → encoding ordinal
mapping_mental = {
    'Poor': 0,
    'Fair': 1,
    'Good': 2,
    'Excellent': 3,
    'Other': -1
}
data['mental_health_status'] = data['mental_health_status'].map(mapping_mental)

# 2.3 Variables binaires
binary_mapping = {'No': 0, 'Yes': 1}
binary_cols = ['support_systems_access', 'online_support_usage']

for col in binary_cols:
    data[col] = data[col].map(binary_mapping)

# ================================
# 🔥 3. CORRECTION IMPORTANTE (occupation)
# ================================

# Regrouper les catégories rares
data['occupation'] = data['occupation'].replace({
    'Student': 'Other',
    'Self-employed': 'Other'
})

# One-Hot Encoding optimisé
data = pd.get_dummies(data, columns=['occupation'], prefix='occ')

# 🔥 convertir True/False → 0/1
data = data.astype({col: 'int' for col in data.columns if col.startswith('occ_')})

# ================================
# 4. Suppression des colonnes inutiles
# ================================
cols_to_drop = [
    'social_comparison_trigger', 
    'user_id',
    'gad7_score', 
    'gad7_severity', 
    'phq9_score', 
    'phq9_severity',
    'work_environment_impact',
    'leisure_screen_hours',
    'work_mode',
    'diet_quality',
    'user_archetype',
    'primary_platform',
    'activity_type',
    'dominant_content_type',
    'late_night_usage'
]

data = data.drop(columns=cols_to_drop)

# ================================
# 5. Vérification finale
# ================================
print(f"Nouvelle forme du dataset : {data.shape}")

# Voir les colonnes créées pour occupation
print(data.filter(like='occ_').sum())

data.head()

Nouvelle forme du dataset : (38400, 24)
occ_Employed      38207
occ_Other           152
occ_Retired          14
occ_Unemployed       27
dtype: int64


,age,gender,screen_time_hours,sleep_hours,stress_level,tech_usage_hours,social_media_hours,gaming_hours,mental_health_status,physical_activity_hours,...,mood_level,work_screen_hours,sleep_quality,exercise_minutes_per_week,social_hours_per_week,mental_wellness_index,occ_Employed,occ_Other,occ_Retired,occ_Unemployed
0,23.0,0,12.36,8.01,Low,6.57,6.00,0.68,2,6.71,...,5.16,1.455,1.0,103.0,7.75,14.8,1,0,0,0
1,21.0,1,7.61,7.28,High,3.01,2.57,3.74,0,5.88,...,5.16,1.455,1.0,103.0,7.75,14.8,1,0,0,0
2,51.0,1,3.16,8.04,High,3.04,6.14,1.26,1,9.81,...,5.16,1.455,1.0,103.0,7.75,14.8,1,0,0,0
3,25.0,0,13.08,5.62,Medium,3.84,4.48,2.59,3,5.28,...,5.16,1.455,1.0,103.0,7.75,14.8,1,0,0,0
4,53.0,1,12.63,5.55,Low,1.20,0.56,0.29,2,4.00,...,5.16,1.455,1.0,103.0,7.75,14.8,1,0,0,0
